# 19.17 Auto 怎樣選檔，手動又怎樣優先？


現在把機率判斷器接到 Looped 解題器：先判斷題目是否屬於支援任務、必要資訊是否完整，再選 Looped 解題器要跑的輪數。這是一個服務的 **Auto effort**，也就是由服務自動選[7.23](https://birdhackor.github.io/tiny-perceptron-vlm/7.23.html)的思考投入檔位；Looped 解題器仍負責解題，機率判斷器提供可觀察的判斷，決策規則把判斷變成預算。三者不必被混成同一種模型能力。

接上 Looped 解題器時，機率判斷器也要學會辨認 Looped 解題器的材料。同一個「需要哪類服務？」問題，候選擴為卡片、地址、App、兩盒追蹤、三數加法及不適用；相關題目也要進入訓練與驗證，依[19.14](https://birdhackor.github.io/tiny-perceptron-vlm/19.14.html)繼承共同家族切分。不能只在使用時臨時加入候選，就當作機率判斷器已理解新含義。

資訊充分也按這個有限介面的欄位契約判斷：追蹤題要求兩盒初值、事件序列與問哪一盒，加法要求三個運算元。例如 `兩盒追蹤;A=2;B=6;事件=交換;查詢=B` 材料齊全；把 A 的初值換成 `?`（寫成 `A=?`）後，B的最後值便無法確定，機率判斷器應判資訊不足，先詢問再交給 Looped 解題器。標籤依世界規則準備，不先讓 Looped 解題器猜答案再反推是否充分。客服、追蹤與加法的充分性各按自己的流程要求，不能拿「有沒有申請編號」通用於所有題目。

使用者可傳 `auto/low/medium/high`。指定固定檔時，服務用已凍結的對應設定；auto時才依決策規則選檔。無論選哪一種，都先檢查支援範圍與硬上限。若某個檔超出使用者上限，要依介面約定回報受限或拒絕，不能悄悄加算；也不能選了low卻偷偷跑high，再把結果算成low。

第一版使用容易查清楚的**規則 Auto**，整個流程分三步：

1. 機率判斷器各作一次 Choice 與資訊充分判斷，以當前候選中分數最高者決定，不另加機率門檻。
2. Choice 選的不是兩盒追蹤或三數加法，就回 `unsupported`；資訊不齊，就回 `clarification_needed`。這兩種情況都不呼叫 Looped 解題器。
3. 材料齊全且任務合適，才決定輪數交給 Looped 解題器：手動檔位直接查表，auto 時查下面這條凍結規則。

規則本身只有十行，下面是函式庫的原始碼：


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# 節錄自 tiny_perceptron/lesson_products/inference.py：rule_depth
def rule_depth(policy, task, event_count=0):
    if policy.get("version") != 1:
        raise ValueError("需要凍結的Auto規則版本")
    if task == "sum":
        depth = policy["sum_depth"]
    else:
        depth = 1 if event_count <= policy["low_max_events"] else 3 if event_count <= policy["medium_max_events"] else 5
    if depth not in (1, 3, 5):
        raise ValueError("Auto規則只能選1／3／5輪")
    return depth

`rule_depth` 收三個東西：凍結的規則 `policy`、任務 `task`，以及事件數 `event_count`。第一個 `if` 確認規則版本是 1，避免拿到格式不同的設定。加法題直接用 `sum_depth`；追蹤題用一行接連的條件運算式：事件數不超過 `low_max_events` 就跑 1 輪，不超過 `medium_max_events` 就跑 3 輪，否則跑 5 輪。最後再確認結果只會是 1、3、5。


In [ ]:
from tiny_perceptron.lesson_products.inference import EFFORT

policy = {"version": 1, "low_max_events": 0, "medium_max_events": 4, "sum_depth": 1}
for events in range(1, 7):
    print("兩盒追蹤", events, "個事件 → auto 選", rule_depth(policy, "boxes", events), "輪")
print("三數加法 → auto 選", rule_depth(policy, "sum"), "輪")
print("手動檔位", EFFORT)
print("指定 medium、硬上限 2 → 實跑", min(EFFORT["medium"], 2), "輪")

`policy` 的四個值抄自凍結檔 [b-auto-policy.json](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/lesson-products/results/b-auto-policy.json)（看 `low_max_events`、`medium_max_events`、`sum_depth` 三欄）；公開的 Looped 解題器包裡的 `auto-policy.json` 和它的 SHA-256 相同。`low_max_events` 是 0，而追蹤題至少有 1 個事件，所以 auto 從不替追蹤題選 1 輪：輸出顯示 1–4 個事件選 3 輪，5–6 個事件選 5 輪，三數加法固定 1 輪。`EFFORT` 是手動檔位表，low、medium、high 對應 1、3、5 輪。`min(EFFORT["medium"], 2)` 示範硬上限：指定 medium 要求 3 輪，上限 2 就只跑 2 輪；函式庫會把這種結果標成 `budget_limited`，不假裝跑滿。

完整流程寫在 [inference.py](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/tiny_perceptron/lesson_products/inference.py) 的 `auto_solve`：它依序呼叫兩次機率判斷器，`choice["selected"] not in ("boxes", "sum")` 時回 `unsupported`，`info["selected"] != "yes"` 時回 `clarification_needed`；否則算出要求的輪數 `requested`（手動檔位查 `EFFORT`，auto 呼叫 `rule_depth`），再用 `actual = min(requested, max_loops)` 套上硬上限。

本次在決策資料上比較了45組事件長度界線及加法輪數，目標是在完整題目平均輪數不超過3的條件下，讓整個流程答對率最高；選出的就是上面這組。主要測試題的追蹤題全是 1–4 個事件，因此全部用3輪，這次並未替它們找到更細的分檔。5–6個事件會選5輪，但仍屬壓力測試，沒有因此變可靠。

設定凍結後，Auto在2998個完整題中答對2926題；固定medium是2927題。Auto只把42道加法各少算2輪，共少84輪，也少對1題。這是這份測試題的實際品質／工作量取捨，沒有證明Auto優於固定medium。

決策規則的目標也要先明定，例如「在平均解題輪數不超過3的限制內，讓答對率最高」。機率判斷器的計算不能被免費略過；實驗紀錄另外量它的時間，總時間包含機率判斷器與 Looped 解題器。若題目很簡單，機率判斷器的開銷可能比直接用 low 跑 Looped 解題器還大；若機率判斷器錯估難度，Auto也可能輸給固定medium。

更進一步可學各預算的成功率或加算收益，但需要**新標籤**。先凍結 Looped 解題器、tokenizer、提示、讀出答案的方式與預算定義，再在路由訓練題跑各檔、記成敗；不能把Choice的集中度直接當成 Looped 解題器會答對的機率。單次0／1只是一筆結果；若讀出答案時用抽樣（[1.15](https://birdhackor.github.io/tiny-perceptron-vlm/1.15.html)）而不是取最高分，還得重複多次才能估成功率。

進階路由訓練後還要重驗機率判斷器原本的選項與校準能力；Looped 解題器一改，舊路由標籤也可能失效。因此這一版先交付規則Auto，學習式Auto另列後續里程碑。第一版是開始前選輪數，不聲稱中途可無成本改檔或續算；重新從low跑high時，兩次成本都要記。

練習把第二段程式的 `"medium_max_events": 4` 改成 2，先預測 3、4 個事件改選 5 輪，再執行核對；這樣平均輪數會超過 3，不符合本次的決策目標。另一題：題目少了必要起始狀態，選high能解決嗎？更多計算不能補出未提供的事實；先澄清，再把新材料交給 Looped 解題器，才符合這個流程。
